In [ ]:
import os
import numpy as np
import pandas as pd
import pyarrow as pa

_NCPU = os.cpu_count() or 1
pa.set_cpu_count(_NCPU)
pa.set_io_thread_count(_NCPU)
os.environ["NUMEXPR_MAX_THREADS"] = str(_NCPU)
os.environ["NUMEXPR_NUM_THREADS"] = str(_NCPU)
os.environ.setdefault("OMP_NUM_THREADS", str(_NCPU))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_NCPU))
os.environ.setdefault("MKL_NUM_THREADS", str(_NCPU))
print(f"Running with {_NCPU} CPU cores | pyarrow {pa.__version__}", flush=True)

In [ ]:
def _add_pasa_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Features derived from PDPASA_REGIONSOLUTION 24h-ahead availability forecast.

    pasa_avail_gen = AEMO forecast of total available generation 24h from now.
    This captures scheduled outages — a generator booked off for maintenance
    tomorrow — which is the primary driver of predictable price spikes.

    No-op if pasa_avail_gen is absent or all-NaN.
    """
    df = df.copy()
    if "pasa_avail_gen" not in df.columns or df["pasa_avail_gen"].isna().all():
        return df

    pa   = df["pasa_avail_gen"]
    dem50 = df["pasa_demand50"] if "pasa_demand50" in df.columns else pd.Series(np.nan, index=df.index)

    # Direct 24h-ahead signals
    df["pasa_avail_gen"]  = pa.astype(np.float32)
    df["pasa_demand50"]   = dem50.astype(np.float32)

    # Forward reserve margin: spare capacity AEMO expects 24h from now
    df["pasa_margin"]     = (pa - dem50).clip(-5000, 10000).astype(np.float32)
    df["pasa_util"]       = (dem50 / (pa + 1)).clip(0, 2).astype(np.float32)

    # Deviation from current available gen (measures scheduled outage delta)
    if "avail_gen" in df.columns:
        df["pasa_vs_now"]     = (pa - df["avail_gen"]).clip(-8000, 8000).astype(np.float32)
        df["pasa_vs_now_pct"] = ((pa - df["avail_gen"]) / (df["avail_gen"] + 1)).clip(-1, 1).astype(np.float32)

    # Demand surprise vs PASA50: how much does actual demand deviate from 24h-ahead
    # PASA demand forecast? Positive = demand exceeded the forecast = supply tighter.
    if "demand" in df.columns and not dem50.isna().all():
        d_vs_pasa = (df["demand"] - dem50).clip(-3000, 3000)
        df["demand_vs_pasa50"]      = d_vs_pasa.astype(np.float32)
        df["demand_vs_pasa50_abs"]  = d_vs_pasa.abs().astype(np.float32)
        df["demand_vs_pasa50_rm48"] = d_vs_pasa.rolling(48, min_periods=1).mean().astype(np.float32)
        # PASA utilisation: how loaded will system be 24h from now vs PASA forecast?
        df["pasa_demand_ratio"] = (df["demand"] / (dem50 + 1)).clip(0.5, 2.0).astype(np.float32)

    # Rolling trend: is PASA forecast availability declining over next day?
    # min_periods=1 ensures partial windows produce a value rather than NaN,
    # which is critical when pasa_avail_gen has data gaps (ffill is limited).
    df["pasa_avail_rmean_4"]  = pa.rolling(4,  min_periods=1).mean().astype(np.float32)
    df["pasa_avail_rmean_48"] = pa.rolling(48, min_periods=1).mean().astype(np.float32)
    df["pasa_avail_rmin_48"]  = pa.rolling(48, min_periods=1).min().astype(np.float32)
    df["pasa_avail_trend_48"] = (pa - pa.shift(48)).astype(np.float32)  # 24h trend in PASA forecast

    # Low-availability flag (scheduled outage bringing generation below safe floor)
    df["pasa_avail_low"]          = (pa < 8000).astype(np.float32)
    df["pasa_avail_low_count_48"] = df["pasa_avail_low"].rolling(48, min_periods=1).sum().astype(np.float32)

    # Reserve condition code from PDPASA (0=normal, >0=reserve concern)
    if "pasa_reserve_cond" in df.columns:
        df["pasa_reserve_cond"] = df["pasa_reserve_cond"].fillna(0).astype(np.float32)
        df["pasa_low_reserve"]  = (df["pasa_reserve_cond"] > 0).astype(np.float32)

    # Fill any remaining NaN in pasa-derived features with 0 so data gaps
    # don't silently drop rows in the downstream dropna step.
    _pasa_derived = [
        "pasa_margin", "pasa_util", "pasa_vs_now", "pasa_vs_now_pct",
        "pasa_avail_rmean_4", "pasa_avail_rmean_48", "pasa_avail_rmin_48",
        "pasa_avail_trend_48", "pasa_avail_low", "pasa_avail_low_count_48",
        "pasa_low_reserve",
    ]
    for col in _pasa_derived:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    return df